In [92]:
import pandas as pd
import numpy as np
import logging
from sklearn.decomposition import FactorAnalysis, PCA
from sklearn.preprocessing import QuantileTransformer, StandardScaler

# --- Setup Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
logger = logging.getLogger(__name__)

def engineer_and_select_factor(
    df: pd.DataFrame, 
    feature_cols: list, 
    factor_name: str, 
    target: str = 'age_months', 
    threshold: float = 0.65, 
    n_factors: int = 5
):
    """
    Core Engine: Rank-normalizes features, runs EFA, selects the highest 
    'Purity Ratio' factor based on target correlations, and aligns it with growth.
    """
    df = df.copy()
    
    logger.info(f"Processing '{factor_name}': {len(feature_cols)} features.")
    X = df[feature_cols].copy()
    X = X.fillna(X.median())
    
    actual_n_factors = min(n_factors, len(feature_cols))
    
    qt = QuantileTransformer(output_distribution='normal', random_state=42)
    X_ranked = qt.fit_transform(X)
    
    fa = FactorAnalysis(n_components=actual_n_factors, random_state=42)
    factors = fa.fit_transform(X_ranked)
    
    loadings_df = pd.DataFrame(
        fa.components_.T, 
        index=feature_cols, 
        columns=[f'Factor_{i+1}' for i in range(actual_n_factors)]
    )
    
    correlations = df[feature_cols + [target]].corr()[target].drop(target)
    
    current_thresh = threshold
    anchors = correlations[correlations.abs() >= current_thresh].index.tolist()
    while not anchors and current_thresh > 0.3:
        current_thresh -= 0.1
        anchors = correlations[correlations.abs() >= current_thresh].index.tolist()
        
    if not anchors:
        logger.warning(f"No anchors found for {factor_name} even at 0.3 threshold. Fallback to Factor_1.")
        best_factor = 'Factor_1'
        purity_ratio = 1.0
        mean_anchor_loadings = pd.Series({best_factor: np.nan})
        mean_non_anchor_loadings = pd.Series({best_factor: np.nan})
    else:
        non_anchors = [f for f in feature_cols if f not in anchors]
        abs_loadings = loadings_df.abs()
        
        mean_anchor_loadings = abs_loadings.loc[anchors].mean(axis=0)
        
        if non_anchors:
            mean_non_anchor_loadings = abs_loadings.loc[non_anchors].mean(axis=0)
            purity_ratios = mean_anchor_loadings / (mean_non_anchor_loadings + 1e-9)
        else:
            purity_ratios = mean_anchor_loadings 
            
        best_factor = purity_ratios.idxmax()
        purity_ratio = purity_ratios[best_factor]

        print(f"\n{'='*40}")
        print(f"BLOCK: {factor_name.upper()}")
        print(f"{'='*40}")
        print(f"Anchors ({len(anchors)} features > {current_thresh:.2f} r with {target}):")
        print(f"  {anchors[:5]}... (showing up to 5)")
        print(f"\nSelection: {best_factor} (Purity Ratio: {purity_ratio:.2f}x)")
        print(f"Mean Anchor Loading: {mean_anchor_loadings[best_factor]:.3f}")
        if non_anchors:
            print(f"Mean Non-Anchor Loading: {mean_non_anchor_loadings[best_factor]:.3f}")
        print("-" * 40)
        print(f"Anchor Weights on {best_factor}:")
        print(loadings_df.loc[anchors, best_factor].to_string())
        print("=" * 40 + "\n")

    factor_idx = int(best_factor.split('_')[1]) - 1
    df[factor_name] = factors[:, factor_idx]
    
    if df[factor_name].corr(df[target]) < 0:
        df[factor_name] *= -1
        logger.info(f"Flipped {factor_name} to align positively with {target}.")
        
    return df, factor_name


def compute_equal_weighted_mega_factor(df: pd.DataFrame, block_factors: list, target: str = 'age_months') -> tuple:
    """
    Computes a Mega Factor by standardizing and taking the unweighted average of the block factors.
    """
    col_name = "MEGA_FACTOR_AVG"
    logger.info("Generating MEGA FACTOR via Equal-Weighted Average.")
    
    scaler = StandardScaler()
    scaled_blocks = scaler.fit_transform(df[block_factors])
    
    df[col_name] = scaled_blocks.mean(axis=1)
    
    if df[col_name].corr(df[target]) < 0:
        df[col_name] *= -1
        
    return df, col_name


def compute_pca_mega_factor(df: pd.DataFrame, block_factors: list, target: str = 'age_months') -> tuple:
    """
    Computes a Mega Factor using PCA on the block factors, selecting the PC most correlated with age.
    """
    col_name = "MEGA_FACTOR_PCA"
    logger.info("Generating MEGA FACTOR via PCA Selection.")
    
    scaler = StandardScaler()
    scaled_blocks = scaler.fit_transform(df[block_factors])
    
    pca = PCA(n_components=len(block_factors), random_state=42)
    pcs = pca.fit_transform(scaled_blocks)
    
    best_pc_idx = 0
    best_corr = 0
    
    # Check correlation of each PC with target to find the max absolute correlation
    target_series = df[target].reset_index(drop=True)
    for i in range(pcs.shape[1]):
        corr = pd.Series(pcs[:, i]).corr(target_series)
        if abs(corr) > abs(best_corr):
            best_corr = corr
            best_pc_idx = i
            
    df[col_name] = pcs[:, best_pc_idx]
    
    print(f"\n{'='*40}")
    print(f"BLOCK: {col_name}")
    print(f"{'='*40}")
    print(f"Selected PC_{best_pc_idx + 1} with absolute age correlation of {abs(best_corr):.3f}")
    
    # Show PCA loadings for transparency
    loadings = pd.Series(pca.components_[best_pc_idx], index=block_factors)
    print("PCA Loadings on selected component:")
    print(loadings.to_string())
    print("=" * 40 + "\n")
    
    if df[col_name].corr(df[target]) < 0:
        df[col_name] *= -1
        
    return df, col_name


def run_full_hierarchical_pipeline(
    df: pd.DataFrame, 
    block_prefixes: dict, 
    target: str = 'age_months', 
    mlu_col: str = 'mlu',
    base_threshold: float = 0.65
):
    """
    Executes all block-level reductions and generates 4 variants of the Mega Factor 
    for comparison.
    """
    processed_df = df.copy()
    generated_block_factors = []
    all_raw_features = []
    
    # --- PHASE 1: Block Level Factors ---
    for block_name, prefix in block_prefixes.items():
        
        block_features = [
            col for col in processed_df.columns 
            if str(col).startswith(prefix) and pd.api.types.is_numeric_dtype(processed_df[col])
        ]
        
        if not block_features:
            logger.warning(f"No numeric features found for prefix '{prefix}'. Skipping.")
            continue
            
        all_raw_features.extend(block_features)
        block_score_name = f"{block_name}_block_factor"
        
        processed_df, final_col_name = engineer_and_select_factor(
            df=processed_df,
            feature_cols=block_features,
            factor_name=block_score_name,
            target=target,
            threshold=base_threshold,
            n_factors=5 
        )
        generated_block_factors.append(final_col_name)
        
    # Variables to track generated mega factors
    mega_factors = []

    if len(generated_block_factors) > 1:
        # --- PHASE 2A: Two-Stage EFA Mega Factor ---
        logger.info("Generating MEGA FACTOR via Two-Stage EFA.")
        processed_df, m_efa_2stage = engineer_and_select_factor(
            df=processed_df,
            feature_cols=generated_block_factors,
            factor_name="MEGA_FACTOR_2STAGE_EFA",
            target=target,
            threshold=base_threshold - 0.15, 
            n_factors=min(3, len(generated_block_factors))
        )
        mega_factors.append(m_efa_2stage)

        # --- PHASE 2B: One-Stage EFA Mega Factor ---
        if all_raw_features:
            logger.info("Generating MEGA FACTOR via One-Stage EFA (Flat).")
            processed_df, m_efa_1stage = engineer_and_select_factor(
                df=processed_df,
                feature_cols=all_raw_features,
                factor_name="MEGA_FACTOR_1STAGE_EFA",
                target=target,
                threshold=base_threshold, 
                n_factors=min(10, len(all_raw_features) // 3) # Allow more factors for flat dataset
            )
            mega_factors.append(m_efa_1stage)

        # --- PHASE 2C: Equal-Weighted Average Mega Factor ---
        processed_df, m_avg = compute_equal_weighted_mega_factor(
            df=processed_df, 
            block_factors=generated_block_factors, 
            target=target
        )
        mega_factors.append(m_avg)

        # --- PHASE 2D: PCA-derived Mega Factor ---
        processed_df, m_pca = compute_pca_mega_factor(
            df=processed_df, 
            block_factors=generated_block_factors, 
            target=target
        )
        mega_factors.append(m_pca)
        
    else:
        logger.error("Not enough blocks generated to create Mega Factors.")

    # --- PHASE 3: Giant Correlation Matrix ---
    print("\n" + "#"*60)
    print("FINAL EVALUATION: GIANT CORRELATION MATRIX (Target & MLU)")
    print("#"*60)
    
    eval_cols = [target]
    if mlu_col in processed_df.columns:
        eval_cols.append(mlu_col)
        
    eval_cols.extend(mega_factors)
    eval_cols.extend(generated_block_factors)
    
    corr_matrix = processed_df[eval_cols].corr()
    
    # Filter the matrix to only show rows for the variables, and columns for Age/MLU
    # Sorted by correlation with Age descending
    target_matrix = corr_matrix[[target] + ([mlu_col] if mlu_col in processed_df.columns else [])]
    sorted_target_matrix = target_matrix.sort_values(by=target, ascending=False)
    
    print(sorted_target_matrix.round(3).to_string())
    
    return processed_df, corr_matrix

In [95]:
# ==========================================
# --- EXECUTION EXAMPLE ---
# ==========================================

# --- Execution ---
master_df = pd.read_csv('../data/old/master_features.csv')

exclude_cols = [
        'child_id', 'age_months', 'mlu', 'ttr', 
        'mixture_cluster', 'sess_id', 'exp', 'lang'
    ]
    
feature_cols = [
    col for col in master_df.columns 
    if col not in exclude_cols and pd.api.types.is_numeric_dtype(master_df[col])
]

rename_dict = {
    col: f"lex_{col}" 
    for col in master_df.columns 
    if col in feature_cols
}

# Use the .rename() method (This updates the internal Indexer/Metadata)
master_df = master_df.rename(columns=rename_dict)

# Define your blocks by their string prefixes
blocks_to_process = {
    'lexical': 'lex_'
}

final_df, final_corr_matrix = run_full_hierarchical_pipeline(
    df=master_df,
    block_prefixes=blocks_to_process,
    target='age_months',
    mlu_col='mlu',
    base_threshold=0.65
)

2026-03-19 00:22:45,194 - [INFO] - Processing 'lexical_block_factor': 29 features.
/Users/joshuacastillo/Desktop/BSE_DSDM/T2/NLP/final_project/developmental-patterns-child-speech/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (1000) is greater than the total number of samples (214). n_quantiles is set to n_samples.
  warnings.warn(
2026-03-19 00:22:45,252 - [ERROR] - Not enough blocks generated to create Mega Factors.



BLOCK: LEXICAL_BLOCK_FACTOR
Anchors (3 features > 0.65 r with age_months):
  ['lex_svd_2', 'lex_score_late_complex', 'lex_topic_late_complex']... (showing up to 5)

Selection: Factor_3 (Purity Ratio: 1.73x)
Mean Anchor Loading: 0.828
Mean Non-Anchor Loading: 0.480
----------------------------------------
Anchor Weights on Factor_3:
lex_svd_2                -0.853768
lex_score_late_complex    0.845734
lex_topic_late_complex    0.783515


############################################################
FINAL EVALUATION: GIANT CORRELATION MATRIX (Target & MLU)
############################################################
                      age_months    mlu
age_months                 1.000  0.667
mlu                        0.667  1.000
lexical_block_factor       0.660  0.850
